# Modelos avançados no Kaggle

Este notebook **não roda no ambiente local** — foi gerado aqui, mas é
feito para ser subido e executado no Kaggle, onde PyTorch + GPU já vêm
prontos (ver `reports/tables/modelos_avancados_smoke_test.md` para o
porquê de não dar para rodar isso localmente).

Segue a mesma organização dos outros notebooks deste repositório: o código
fica no pacote `src/churn_telecom/`, e este notebook só clona o repositório,
instala o pacote e orquestra os experimentos.

**No Kaggle, antes de rodar:** ative *Settings → Internet → On* (o
notebook clona o repositório, baixa o CSV e instala os pacotes) e
*Accelerator → GPU*. Depois use **"Save Version" → "Save & Run All
(Commit)"**, que executa num kernel gerenciado em segundo plano.


In [ ]:
#@title Setup (clona o repo e instala o pacote)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/jpbezerra/redes-neurais.git"   # ajuste se necessario
BRANCH   = "feat/churn"   # ajuste aqui se a branch de trabalho mudar
REPO_DIR = "redes-neurais"
SUBPATH  = "projeto/churn-telecom"

def _sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=False)

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

if IN_COLAB or IN_KAGGLE:
    base = Path("/content") if IN_COLAB else Path("/kaggle/temp")  # fora de /kaggle/working: o clone nao entra no Output
    root = base / REPO_DIR
    if root.exists():
        _sh(f"cd {root} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}")
    else:
        _sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {root}")
    PROJECT_ROOT = root / SUBPATH
    _sh(f"pip install -q -e {PROJECT_ROOT}")
else:
    PROJECT_ROOT = Path.cwd().parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# No Kaggle, tudo que for gerado vai para /kaggle/working/saida (so isso aparece
# no Output, em vez do repositorio inteiro).
OUT_DIR = Path("/kaggle/working/saida") if IN_KAGGLE else PROJECT_ROOT

DATA_DIR    = PROJECT_ROOT / "data"
RESULTS_DIR = OUT_DIR / "results"
TABLES_DIR  = OUT_DIR / "reports" / "tables"
for d in (DATA_DIR, RESULTS_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR :", RESULTS_DIR)


## Instalação dos modelos avançados

O Kaggle já vem com PyTorch e a maioria das libs de ML — só falta instalar
os 5 modelos em si.

In [2]:
!pip install -q tabpfn pykan tabkan
!pip install -q "autogluon.tabular[mitra]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.5/827.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.1 MB/s eta 0:00:00


## Preparação dos dados

`load_raw` baixa o CSV automaticamente se não existir em `DATA_DIR`
(mesma conveniência do mini-projeto 2) — não é preciso subir nada como
Kaggle Dataset separado, o clone do repositório já basta.

In [3]:
import torch
import joblib
import numpy as np
import pandas as pd

from churn_telecom.data import load_raw, split_three_stage, assert_sem_vazamento
from churn_telecom.features import Preprocessor
from churn_telecom.metrics import full_report
from churn_telecom.checkpointing import save_run, load_all_metadata

SEED = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

df = load_raw(DATA_DIR)
split = split_three_stage(df, seed=SEED)
assert_sem_vazamento(split)
df_train, df_val, df_test = df.iloc[split.train_idx], df.iloc[split.val_idx], df.iloc[split.test_idx]

prep = Preprocessor().fit(df_train)
X_train, cols = prep.transform(df_train)
X_val, _ = prep.transform(df_val)
X_test, _ = prep.transform(df_test)
y_train = prep.transform_target(df_train)
y_val = prep.transform_target(df_val)
y_test = prep.transform_target(df_test)

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")


device: cuda
Baixando CSV de https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv ...
CSV salvo em /kaggle/working/redes-neurais/projeto/churn-telecom/data/telco_customer_churn.csv
X_train: (5174, 40) | X_val: (2588, 40) | X_test: (1761, 40)


## Execução isolada por modelo

Cada modelo roda dentro de `rodar(nome, fn)`: se um falhar (pacote,
download de pesos, memória...), o erro é registrado em `FALHAS` e o
notebook **continua** para os próximos — um "Save & Run All" nunca mais
morre no meio por causa de um único modelo. As falhas são salvas em
`reports/tables/modelos_avancados_falhas_kaggle.csv` no final.

In [4]:
import traceback

FALHAS = {}

def rodar(nome, fn):
    try:
        fn()
        print(f"[ok] {nome}")
    except Exception as e:
        FALHAS[nome] = f"{type(e).__name__}: {e}"
        print(f"[FALHOU] {nome}: {type(e).__name__}: {e}")
        traceback.print_exc()


## STab (implementação própria — ver `src/churn_telecom/models/stab.py`)

Não existe um pacote público legítimo chamado "STab" (colisão de nome no
PyPI — ver smoke test). Esta é uma implementação própria e enxuta de
Transformer para dados tabulares, cobrindo o espaço de hiperparâmetros
pedido no enunciado.

In [5]:
from churn_telecom.models.stab import STabHyperparams, treinar_stab, prever_stab

def experimento_stab():
    hp = STabHyperparams(dim=32, depth=2, heads=4, attn_dropout=0.1, ff_dropout=0.1,
                         lr=1e-3, weight_decay=1e-5, batch_size=64, max_epochs=200, patience=10, seed=SEED)
    modelo, historico = treinar_stab(X_train, y_train, X_val, y_val, hp, device=device, verbose=True)
    metricas = full_report(y_test, prever_stab(modelo, X_test, device=device))
    print(metricas)
    save_run(RESULTS_DIR, "stab_churn", framework="PyTorch (STab — implementacao propria)",
             hyperparameters=hp.__dict__, metrics=metricas,
             architecture={"dim": hp.dim, "depth": hp.depth, "heads": hp.heads},
             history=historico, notes="STab (implementacao propria).",
             save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))

rodar("stab", experimento_stab)


  epoca    0 | val_loss=0.6926 falhas=0/10
  epoca   10 | val_loss=0.4828 falhas=4/10
{'ks': 0.52237392666627, 'mse': 0.16073084193386022, 'cross_entropy': 0.4819666143591076, 'auroc': 0.8395262789114297, 'threshold': 0.5, 'tp': 352, 'tn': 974, 'fp': 319, 'fn': 116, 'precision': 0.5245901639344263, 'recall': 0.7521367521367521, 'f1': 0.6180860403863037}
[ok] stab


## TabPFN v2

A versão é fixada em **v2** (a do enunciado). Versões novas do pacote usam
por padrão checkpoints v2.5+, que exigem aceitar licença e um
`TABPFN_TOKEN` — inviável num "Save & Run All". Os pesos v2 baixam direto.

In [6]:
from churn_telecom.models.tabpfn import TabPFNHyperparams, treinar_tabpfn, prever_tabpfn

def experimento_tabpfn():
    hp = TabPFNHyperparams(device=device, n_estimators=8, versao="v2", seed=SEED)
    modelo = treinar_tabpfn(X_train, y_train, hp)
    metricas = full_report(y_test, prever_tabpfn(modelo, X_test))
    print(metricas)
    save_run(RESULTS_DIR, "tabpfn_churn", framework="TabPFN v2", hyperparameters=hp.__dict__,
             metrics=metricas, notes="TabPFN v2 (checkpoint v2, sem token).",
             save_model_fn=lambda d: joblib.dump(modelo, d / "model.joblib"))

rodar("tabpfn", experimento_tabpfn)


tabpfn-v2-classifier-finetuned-zk73skhh.(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

{'ks': 0.4967973506256569, 'mse': 0.15810432822477177, 'cross_entropy': 0.5095220307990747, 'auroc': 0.8166681539651377, 'threshold': 0.5, 'tp': 189, 'tn': 1182, 'fp': 111, 'fn': 279, 'precision': 0.63, 'recall': 0.40384615384615385, 'f1': 0.4921875}
[ok] tabpfn


## KAN (pykan)

In [7]:
from churn_telecom.models.kan import KANHyperparams, treinar_kan, prever_kan

def experimento_kan():
    hp = KANHyperparams(hidden_width=10, grid=5, k=3, lr=1e-2, steps=200, seed=SEED)
    modelo, historico = treinar_kan(X_train, y_train, X_val, y_val, hp, device=device)
    metricas = full_report(y_test, prever_kan(modelo, X_test, device=device))
    print(metricas)
    save_run(RESULTS_DIR, "kan_churn", framework="pykan (KAN)", hyperparameters=hp.__dict__,
             metrics=metricas, architecture={"hidden_width": hp.hidden_width, "grid": hp.grid, "k": hp.k},
             history=pd.DataFrame(historico), notes="KAN.",
             save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))

rodar("kan", experimento_kan)


checkpoint directory created: ./model
saving model version 0.0


| train_loss: nan | test_loss: nan | reg: nan | : 100%|███████████| 200/200 [00:02<00:00, 68.36it/s]


saving model version 0.1
{'ks': 0.03905150018839115, 'mse': nan, 'cross_entropy': nan, 'auroc': nan, 'threshold': 0.5, 'tp': 0, 'tn': 1293, 'fp': 0, 'fn': 468, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
[ok] kan


## TabKAN (ChebyshevKAN do pacote `tabkan`)

O pacote `tabkan` expõe módulos PyTorch (`ChebyshevKAN`, `FourierKAN`, ...),
não um classificador sklearn-like — o wrapper usa `ChebyshevKAN` com
treino Adam + early stopping na validação, igual ao STab.

In [8]:
from churn_telecom.models.tabkan import TabKANHyperparams, treinar_tabkan, prever_tabkan

def experimento_tabkan():
    hp = TabKANHyperparams(hidden_width=10, degree=3, lr=1e-3, batch_size=64,
                           max_epochs=200, patience=10, seed=SEED)
    modelo, historico = treinar_tabkan(X_train, y_train, X_val, y_val, hp, device=device)
    metricas = full_report(y_test, prever_tabkan(modelo, X_test, device=device))
    print(metricas)
    save_run(RESULTS_DIR, "tabkan_churn", framework="tabkan (ChebyshevKAN)", hyperparameters=hp.__dict__,
             metrics=metricas, architecture={"hidden_width": hp.hidden_width, "degree": hp.degree},
             history=historico, notes="TabKAN (ChebyshevKAN).",
             save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))

rodar("tabkan", experimento_tabkan)


  epoca    0 | val_loss=0.5193 falhas=0/10
  epoca   10 | val_loss=0.4866 falhas=0/10
  epoca   20 | val_loss=0.4858 falhas=8/10
{'ks': 0.5034307018065718, 'mse': 0.16770722127348725, 'cross_entropy': 0.49603115281221277, 'auroc': 0.8342744627547412, 'threshold': 0.5, 'tp': 352, 'tn': 957, 'fp': 336, 'fn': 116, 'precision': 0.5116279069767442, 'recall': 0.7521367521367521, 'f1': 0.6089965397923875}
[ok] tabkan


## Mitra (via AutoGluon)

In [ ]:
from churn_telecom.models.mitra import MitraHyperparams, treinar_mitra, prever_mitra, salvar_mitra

def experimento_mitra():
    hp = MitraHyperparams(time_limit_segundos=600, num_gpus=1 if device == "cuda" else 0, seed=SEED)
    preditor = treinar_mitra(X_train, y_train, cols, hp, path="/kaggle/temp/ag_mitra")
    metricas = full_report(y_test, prever_mitra(preditor, X_test, cols))
    print(metricas)
    save_run(RESULTS_DIR, "mitra_churn", framework="AutoGluon (extra [mitra])",
             hyperparameters=hp.__dict__, metrics=metricas, notes="Mitra.",
             save_model_fn=lambda d: salvar_mitra(preditor, d / "autogluon_predictor"))

rodar("mitra", experimento_mitra)


## Comparação final — todos os modelos do projeto

Funciona mesmo que algum modelo acima tenha falhado.

In [10]:
pd.DataFrame([{"modelo": k, "erro": v} for k, v in FALHAS.items()],
             columns=["modelo", "erro"]).to_csv(TABLES_DIR / "modelos_avancados_falhas_kaggle.csv", index=False)
print("Falhas:", FALHAS or "nenhuma")

resumo = load_all_metadata(RESULTS_DIR)
cols_mostrar = ["model_id", "metrics.ks", "metrics.auroc", "metrics.recall", "metrics.precision"]
cols_existentes = [c for c in cols_mostrar if c in resumo.columns]
tabela = resumo[cols_existentes].sort_values("metrics.ks", ascending=False).reset_index(drop=True)
tabela


Falhas: nenhuma


,model_id,metrics.ks,metrics.auroc,metrics.recall,metrics.precision
0,gb_churn_baseline_oversample,0.524183,0.831954,0.743590,0.532110
1,stab_churn,0.522374,0.839526,0.752137,0.524590
2,xgb_churn_baseline_oversample,0.519855,0.831416,0.752137,0.528529
3,gb_churn_features_charges_per_tenure,0.516083,0.831512,0.747863,0.529501
4,gb_churn_baseline_classweight,0.512136,0.831246,0.758547,0.512266
5,mlp_churn_baseline_oversample,0.511046,0.829421,0.767094,0.493132
6,mlp_churn_optuna_leva1,0.510758,0.816685,0.724359,0.544141
7,gb_churn_optuna_leva1,0.510391,0.835575,0.756410,0.513043
8,xgb_churn_baseline_scaleposweight,0.509767,0.833499,0.767094,0.515805
9,tabkan_churn,0.503431,0.834274,0.752137,0.511628


## Salvando os resultados (sem baixar o repositório inteiro)

Esta célula faz duas coisas:

1. Gera `results_churn.zip` só com `results/` + tabelas (poucos MB) — no
   Output do Kaggle aparece só isso, não o repositório clonado.
2. **Se existir o segredo `GITHUB_TOKEN`** (Kaggle → *Add-ons → Secrets*;
   token do GitHub com permissão *Contents: Read and write* neste repo),
   faz push dos resultados para a branch `results/churn-kaggle`. Localmente:
   `git fetch origin results/churn-kaggle` e
   `git checkout origin/results/churn-kaggle -- projeto/churn-telecom/results projeto/churn-telecom/reports/tables`.

Sem o segredo, só o zip é gerado.

In [ ]:
import shutil

RESULTS_BRANCH = "results/churn-kaggle"

if IN_KAGGLE:
    zip_path = shutil.make_archive("/kaggle/working/results_churn", "zip", OUT_DIR)
    print("Zip gerado:", zip_path)

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None

    if not token:
        print("Sem segredo GITHUB_TOKEN: pulando o push (baixe o zip pelo Output).")
    else:
        # copia as saidas para dentro do clone e commita (results/ e ignorado pelo git -> add -f)
        shutil.copytree(RESULTS_DIR, PROJECT_ROOT / "results", dirs_exist_ok=True)
        shutil.copytree(TABLES_DIR, PROJECT_ROOT / "reports" / "tables", dirs_exist_ok=True)

        def git(*args, secret=False):
            r = subprocess.run(["git", "-C", str(root), "-c", "user.name=kaggle-run",
                                "-c", "user.email=kaggle-run@users.noreply.github.com", *args],
                               capture_output=True, text=True)
            saida = (r.stdout + r.stderr).replace(token, "***")
            if r.returncode != 0:
                raise RuntimeError(f"git {args[0]} falhou: {saida}")
            return saida

        git("checkout", "-B", RESULTS_BRANCH)
        git("add", "-f", f"{SUBPATH}/results", f"{SUBPATH}/reports/tables")
        git("commit", "-m", "results: saidas do notebook de modelos avancados (Kaggle)")
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        git("push", "--force", url, f"HEAD:refs/heads/{RESULTS_BRANCH}")
        print(f"Resultados enviados para a branch {RESULTS_BRANCH}.")
